In [2]:
import torch
import pandas as pd
import pickle
from tqdm import tqdm

import os
import sys
base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from nesy_gnn.models import FuzzyGAT, FuzzyNN
from nesy_gnn.train_model import train_epoch, evaluate, build_pyg_data

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def load_graph(filepath):
    with open(filepath, 'rb') as f:
        G = pickle.load(f)
    print(f"Load graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")
    return G

In [4]:
G = load_graph("../nesy_gnn/results/updated/AD_KG/ANMerge/4/patient_graph.gpickle")
data_all      = build_pyg_data(G, use_direct_rules=False)
data_direct   = build_pyg_data(G, use_direct_rules=True)

Load graph with 691 nodes and 1406 edges
Computing topological features...
Computing topological features...


In [5]:
data_direct

Data(x=[691, 4444], edge_index=[2, 2812], edge_attr=[2812], y=[691], kg_scores=[691, 185], topo_features=[691, 5], topo_scaler_mean=[5], topo_scaler_scale=[5])

In [6]:
data_all

Data(x=[691, 4444], edge_index=[2, 2812], edge_attr=[2812], y=[691], kg_scores=[691, 741], topo_features=[691, 5], topo_scaler_mean=[5], topo_scaler_scale=[5])

In [7]:
data_direct.topo_features.size(1) + data_direct.kg_scores.size(1)

190

In [8]:
model2 = FuzzyNN(in_channels=data_direct.x.size(1), 
                            hidden_channels=64, 
                            out_channels=2, 
                            num_rules=data_direct.topo_features.size(1) + data_direct.kg_scores.size(1),
                            num_layers=2,
                            )
optimizer = torch.optim.AdamW(model2.parameters(), lr=1e-4, weight_decay=1e-5)

In [9]:
model = FuzzyGAT(in_channels=data_direct.x.size(1), 
                            hidden_channels=64, 
                            out_channels=2, 
                            num_rules=190,
                            num_layers=2,
                            heads=4,)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

In [10]:
import numpy as np
from sklearn.model_selection import train_test_split


train_idx, test_idx = train_test_split(np.arange(data_direct.x.size(0)), 
                                       test_size=0.2, 
                                       stratify = data_direct.y, 
                                       random_state=42)
train_mask = torch.zeros(data_direct.x.size(0), dtype=torch.bool)
test_mask = torch.zeros(data_direct.x.size(0), dtype=torch.bool)
train_mask[train_idx]=True
test_mask[test_idx]=True

for _ in tqdm(range(10), desc="Training"):
    loss = train_epoch(model=model2,
                       data=data_direct,
                       train_mask=train_mask,
                       optimizer=optimizer,
                       use_kg=True,
                       use_topo=True)

Training: 100%|██████████| 10/10 [00:00<00:00, 100.29it/s]

Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])
Number of rules: 190
rule features size: torch.Size([691, 190])


In [11]:
fuzzy_params = {
                "theta": getattr(model.fuzzy_layer, "theta", None),
                "alpha": getattr(model.fuzzy_layer, "alpha", None)
            }
cls_weight = model.classifier.weight # [out_dim, hidden_dim]
activation_weight = model.rule_proj.weight # [hidden_dim, num_rules]
activation_weight.size()

torch.Size([64, 190])

In [12]:
result = model2(
            x=data_direct.x,
            edge_index=data_direct.edge_index,
            edge_attr=data_direct.edge_attr,
            rule_features=torch.cat([data_direct.topo_features, data_direct.kg_scores], dim=1)
        )
result

Number of rules: 190
rule features size: torch.Size([691, 190])


{'logits': tensor([[ 0.0006,  0.0823],
         [-0.0606,  0.1051],
         [-0.0888,  0.0856],
         ...,
         [-0.0611,  0.0510],
         [-0.0979,  0.0777],
         [-0.0807,  0.1337]], grad_fn=<AddmmBackward0>),
 'rules': tensor([[0.4941, 0.4541, 0.4388,  ..., 0.3212, 0.3332, 0.3794],
         [0.4941, 0.4541, 0.5514,  ..., 0.4108, 0.2352, 0.3761],
         [0.4941, 0.6219, 0.5166,  ..., 0.2962, 0.3294, 0.3067],
         ...,
         [0.4098, 0.6219, 0.4295,  ..., 0.1861, 0.5880, 0.2346],
         [0.9374, 0.3502, 0.6461,  ..., 0.3139, 0.1399, 0.3186],
         [0.3304, 0.2961, 0.4169,  ..., 0.7469, 0.7295, 0.8545]],
        grad_fn=<SigmoidBackward0>),
 'rule_embedding': tensor([[ 0.0393, -0.1500, -0.2897,  ...,  0.2551, -0.2494, -0.0093],
         [-0.0207, -0.2401, -0.3639,  ...,  0.2703, -0.1732, -0.1078],
         [-0.0033, -0.3628, -0.2848,  ...,  0.3223, -0.2122,  0.0362],
         ...,
         [-0.0245, -0.2998, -0.2483,  ...,  0.2547, -0.1337, -0.1357],
       

In [ ]:
result = model(
            data_direct.x,
            data_direct.edge_index,
            data_direct.edge_attr,
            torch.cat([data_direct.topo_features, data_direct.kg_scores], dim=1)
        )
logits = result["logits"]
r = result["rules"]
e = result["rule_embedding"]
g = result["gate"]

num_classes = logits.size(1)
num_classes

2

In [26]:
g.abs().mean(dim=1).shape

torch.Size([691])

In [22]:
result.keys()

dict_keys(['logits', 'rules', 'rule_embedding', 'gate', 'gnn_embedding', 'fused_embedding'])

In [13]:
def get_rule_params(model, data, rule_feature):

    model.eval()
    model.zero_grad(set_to_none=True)

    result = model(
        x=data.x,
        edge_index=data.edge_index,
        edge_attr=data.edge_attr,
        rule_features=rule_feature
    )

    logits = result["logits"]
    r = result["rules"]
    g = result["gate"]

    # IMPORTANT:
    # Do NOT use torch.no_grad() here.
    # We need the autograd graph.
    assert r.requires_grad

    # ---------------------------------------------------------
    # Rule activation statistics
    # ---------------------------------------------------------
    mean_activation = r.detach().mean(dim=0)
    std_activation = r.detach().std(dim=0)

    activation_rate = (
        (r.detach() > 0.5)
        .float()
        .mean(dim=0)
    )

    # ---------------------------------------------------------
    # Gradient × activation: class 0
    # ---------------------------------------------------------
    score_0 = logits[:, 0].sum()

    grad_0 = torch.autograd.grad(
        score_0,
        r,
        retain_graph=True
    )[0]

    importance_class_0 = (
        r.detach() * grad_0.detach()
    ).abs().mean(dim=0)

    # ---------------------------------------------------------
    # Gradient × activation: class 1
    # ---------------------------------------------------------
    score_1 = logits[:, 1].sum()

    grad_1 = torch.autograd.grad(
        score_1,
        r,
        retain_graph=True
    )[0]

    importance_class_1 = (
        r.detach() * grad_1.detach()
    ).abs().mean(dim=0)

    # ---------------------------------------------------------
    # Gradient × activation: predicted class
    # ---------------------------------------------------------
    predicted_class = logits.detach().argmax(dim=1)

    predicted_score = logits[
        torch.arange(logits.size(0), device=logits.device),
        predicted_class
    ].sum()

    grad_pred = torch.autograd.grad(
        predicted_score,
        r
    )[0]

    importance_prediction = (
        r.detach() * grad_pred.detach()
    ).abs().mean(dim=0)

    # ---------------------------------------------------------
    # Alpha / theta
    # ---------------------------------------------------------
    alpha = model.fuzzy_layer.alpha.detach().cpu()
    theta = model.fuzzy_layer.theta.detach().cpu()

    # ---------------------------------------------------------
    # DataFrame
    # ---------------------------------------------------------
    rule_df = pd.DataFrame({
        "rule": range(len(alpha)),
        "alpha": alpha.numpy(),
        "theta": theta.numpy(),
        "mean_activation": mean_activation.cpu().numpy(),
        "std_activation": std_activation.cpu().numpy(),
        "activation_rate": activation_rate.cpu().numpy(),
        "grad_rule_importance_prediction":
            importance_prediction.cpu().numpy(),
        "importance_class_0":
            importance_class_0.cpu().numpy(),
        "importance_class_1":
            importance_class_1.cpu().numpy(),
    })

    return rule_df, result

In [14]:
num_rules = data_direct.topo_features.size(1) + data_direct.kg_scores.size(1)
rule_feature = torch.cat([data_direct.topo_features, data_direct.kg_scores], dim=1)
rule_feature.size()

torch.Size([691, 190])

In [15]:
rule_feature = torch.cat([data_direct.topo_features, data_direct.kg_scores], dim=1)

df_rule_params2 = get_rule_params(model=model2, 
                                  data=data_direct, 
                                  rule_feature=rule_feature)

Number of rules: 190
rule features size: torch.Size([691, 190])


In [ ]:
df_rule_params = get_rule_params(model=model, data=data_direct, rule_feature)
df_rule_params

,rule,alpha,theta,mean_activation,std_activation,activation_rate,grad_rule_importance_prediction,importance_class_0,importance_class_1
0,0,0.999048,-0.000924,0.487095,0.161220,0.280753,0.006196,0.003002,0.007017
1,1,0.999103,-0.000765,0.483925,0.192823,0.356006,0.004480,0.001384,0.005462
2,2,1.000912,0.000929,0.478777,0.136946,0.178003,0.005352,0.002387,0.005611
3,3,1.000951,0.000913,0.486643,0.161467,0.280753,0.005711,0.015810,0.003506
4,4,0.999124,0.000870,0.526933,0.098360,0.888567,0.012156,0.004443,0.013437
...,...,...,...,...,...,...,...,...,...
185,185,1.000413,-0.000911,0.500914,0.226886,0.493488,0.003392,0.010222,0.002274
186,186,0.999642,0.000903,0.502229,0.213011,0.499276,0.004342,0.001596,0.004871
187,187,0.999106,-0.000205,0.500536,0.225529,0.493488,0.000786,0.001902,0.000705
188,188,0.999734,0.000907,0.501374,0.213525,0.497829,0.008901,0.017907,0.007369


### Permutation Importance

In [38]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)
import torch.nn.functional as F

def test(true, logits):
    pred = logits.argmax(dim=1)
    prob = torch.exp(logits)[:,1]

    metrics = {}
    acc = accuracy_score(true, pred)
    metrics["f1"] = f1_score(true, pred, zero_division=0)
    metrics["f1_macro"] = f1_score(
        true,
        pred,
        average="macro",
        zero_division=0,
    )
    metrics["f1_weighted"] = f1_score(
        true,
        pred,
        average="weighted",
        zero_division=0,
    )

    metrics["auroc"] = roc_auc_score(true, prob)
    
    metrics["auprc"] = average_precision_score(true, prob)
    
    return metrics

@torch.no_grad()
def permutation_importance(model, data, rule_features, labels, metric_name):
    model.eval()

    baseline_out = model(x=data.x, 
        edge_index=data.edge_index, 
        edge_attr=data.edge_attr, 
        rule_features = rule_features)
    baseline_logits = F.log_softmax(baseline_out['logits'], dim=1)
    
    baseline_score = test(labels, baseline_logits)[metric_name]

    importance = []

    for j in range(rule_features.size(1)):
        X_perm = rule_features.clone()

        # Shuffle feature j across nodes
        perm = torch.randperm(X_perm.size(0), device=X_perm.device)
        X_perm[:, j] = X_perm[perm, j]

        out_perm = model(x=data.x, 
                                edge_index=data.edge_index, 
                                edge_attr=data.edge_attr, 
                                rule_features = X_perm)
        logits_perm = F.log_softmax(out_perm['logits'], dim=1)
        
        perm_score = test(labels, logits_perm)[metric_name]

        # Larger drop = more important
        importance.append(baseline_score - perm_score)

    return torch.tensor(importance)

In [14]:
rule_feature = torch.cat(
            [data_direct.topo_features, data_direct.kg_scores],
            dim=1
        )
importance = permutation_importance(model=model,
                                    data=data_direct,
                                    rule_features=rule_feature,
                                    metric_name='auroc',
                                    labels=data_direct.y)

In [16]:
importance.size()

torch.Size([190])

### Intergrated-Gradient based Importance

In [17]:
import torch
from captum.attr import IntegratedGradients

# 1. Create a minimal Wrapper to hide the GNN structure from Captum
class RuleFeatureWrapper(torch.nn.Module):
    def __init__(self, model, x, edge_index, edge_attr=None):
        super().__init__()
        self.model = model
        self.x = x
        self.edge_index = edge_index
        self.edge_attr = edge_attr
        
    def forward(self, rule_features):
        # We only pass rule_features from Captum.
        # The graph structure (x, edge_index) stays static.
        out = self.model(
            x=self.x, 
            edge_index=self.edge_index, 
            edge_attr=self.edge_attr, 
            rule_features=rule_features
        )
        # Captum needs a raw tensor output, so we return the logits
        return out["logits"]

def captum_explain(model, data,rule_feature, targets:torch.Tensor):
    model.eval()
    wrapper = RuleFeatureWrapper(
        model=model, 
        x=data.x, 
        edge_index=data.edge_index, 
        edge_attr=getattr(data, 'edge_attr', None)
    )

    # 3. Initialize Integrated Gradients
    ig = IntegratedGradients(wrapper)

    num_nodes = rule_feature.size(0)

    attributions, delta = ig.attribute(
        inputs=rule_feature,
        target=targets,
        internal_batch_size=num_nodes,  
        n_steps=50, # Optional: number of interpolation steps
        return_convergence_delta=True
    )

    print("Attributions shape:", attributions.shape) # Should be [N, num_rules]
    feature_importance = attributions.abs().mean(dim=0)

    return feature_importance, attributions
    

In [ ]:
feature_importance, attributions = captum_explain(model, data_direct, rule_feature, data_direct.y)

for i, value in enumerate(feature_importance):
    print(f"Feature {i}: {value.item():.6f}")

Attributions shape: torch.Size([691, 190])
Feature 0: 0.001368
Feature 1: 0.001723
Feature 2: 0.000404
Feature 3: 0.001697
Feature 4: 0.000338
Feature 5: 0.001485
Feature 6: 0.001086
Feature 7: 0.001773
Feature 8: 0.002254
Feature 9: 0.000529
Feature 10: 0.002600
Feature 11: 0.001809
Feature 12: 0.001561
Feature 13: 0.000854
Feature 14: 0.001350
Feature 15: 0.001785
Feature 16: 0.002956
Feature 17: 0.001124
Feature 18: 0.000991
Feature 19: 0.001721
Feature 20: 0.000378
Feature 21: 0.001654
Feature 22: 0.002298
Feature 23: 0.002410
Feature 24: 0.000959
Feature 25: 0.003878
Feature 26: 0.002024
Feature 27: 0.001967
Feature 28: 0.003559
Feature 29: 0.001915
Feature 30: 0.003282
Feature 31: 0.002055
Feature 32: 0.001556
Feature 33: 0.001553
Feature 34: 0.000684
Feature 35: 0.002067
Feature 36: 0.001876
Feature 37: 0.002479
Feature 38: 0.001942
Feature 39: 0.000888
Feature 40: 0.001667
Feature 41: 0.001244
Feature 42: 0.002527
Feature 43: 0.003370
Feature 44: 0.002778
Feature 45: 0.002095
F

In [28]:
pd.DataFrame(attributions)

,0,1,2,3,4,5,6,7,8,9,...,180,181,182,183,184,185,186,187,188,189
0,0.000020,0.000362,0.000218,0.000060,0.000349,0.000439,7.885745e-07,0.000670,0.001196,-0.000758,...,-0.000129,0.003572,-0.002504,0.002268,-0.002874,0.000490,0.004244,-0.000010,-0.000270,0.001248
1,0.000019,0.000451,-0.000112,0.000054,0.000284,0.001233,1.306042e-04,0.000772,0.001216,-0.000814,...,0.000086,0.002214,-0.001703,0.001509,-0.003679,0.000014,0.002065,0.000065,-0.000699,0.001216
2,0.000112,0.001478,-0.000027,0.000102,0.000004,0.002376,4.937003e-04,0.001839,0.000835,-0.000031,...,0.003505,-0.002192,-0.000003,0.004622,-0.002931,-0.003909,0.000612,-0.008803,-0.002452,-0.003241
3,0.000100,-0.000610,-0.000007,0.000093,-0.000043,0.001097,1.054507e-04,0.000185,0.000779,0.000056,...,0.001521,-0.002436,0.000181,0.002146,-0.002898,-0.004832,0.003109,-0.006593,-0.004540,-0.001768
4,0.000099,0.004839,-0.000863,0.000099,0.000077,0.001721,2.422064e-05,0.001045,0.000433,0.000436,...,0.002467,-0.001220,0.000231,0.003509,-0.002873,-0.003718,0.001999,-0.011666,-0.001401,-0.003366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
686,0.002621,-0.007197,0.000396,0.003200,-0.000264,-0.000237,-1.838170e-03,-0.001420,-0.005749,0.000203,...,0.001536,-0.002674,0.006329,-0.000177,0.004335,0.001467,0.000163,-0.000430,-0.000008,-0.000482
687,-0.001222,-0.005657,0.000752,-0.001593,-0.000217,-0.001729,4.352468e-03,-0.003571,-0.002975,-0.000282,...,0.002338,-0.001713,0.004726,0.000426,0.001241,0.000156,-0.001145,-0.000553,0.000157,-0.002019
688,0.000361,-0.001132,0.000174,0.000797,0.000201,0.001120,1.929452e-04,0.002303,0.000629,-0.000487,...,0.000535,0.002008,-0.002811,0.001732,-0.001619,-0.000414,-0.000834,0.000146,0.000288,0.002706
689,-0.001606,0.001164,-0.000444,-0.004252,0.000303,0.000277,-2.299219e-05,0.001445,0.001062,-0.000748,...,-0.000107,0.001005,-0.001327,0.001739,-0.001216,0.000294,0.005138,-0.000013,-0.000804,0.001765


## Summarize Rule Importance and Per Sample's Prediction Importance

In [40]:
import torch.nn.functional as F

def explain(model, data, rule_feature,rule_names, patients, metric_name:str):

    # get rule importances
    df, results = get_rule_params(model=model, 
                                            data=data,
                                            rule_feature=rule_feature)
    true_labels = data.y.clone().detach()
    pred_labels = F.log_softmax(results['logits'], dim=1).argmax(dim=1)

    perm_true_importance = permutation_importance(model=model,
                                             data=data,
                                             rule_features=rule_feature,
                                             labels=true_labels,
                                             metric_name=metric_name)
    perm_pred_importance = permutation_importance(model=model,
                                             data=data,
                                             rule_features=rule_feature,
                                             labels=pred_labels,
                                             metric_name=metric_name)
    ig_true_importance, true_attribution = captum_explain(model=model,
                                        data=data,
                                        rule_feature=rule_feature,
                                        targets=true_labels)
    ig_pred_importance, pred_attribution = captum_explain(model=model,
                                            data=data,
                                            rule_feature=rule_feature,
                                            targets=pred_labels)
    df['Permutation_importance(true_label)'] = perm_true_importance
    df['Permutation_importance(pred_label)'] = perm_pred_importance
    df['Integrated_Gradient_importance(true_label)'] = ig_true_importance
    df['Integrated_Gradient_importance(pred_label)'] = ig_pred_importance
    df['rule_name'] = rule_names
    df.set_index('rule_name', inplace=True)

    # create a sample df
    df_samples = pd.DataFrame(pred_attribution)
    df_samples.columns = rule_names
    g_avg = results['gate'].abs().mean(dim=1)
    df_samples['gate'] = g_avg.detach().numpy()
    df_samples['sample_id'] = patients
    df_samples['Label'] = true_labels
    df_samples['Prediction'] = pred_labels
    df_samples.set_index('sample_id', inplace=True)

    return df, df_samples

In [30]:
import json

meta_path = "../nesy_gnn/results/updated/AD_KG/ANMerge/4/graph_meta.json"
with open(meta_path, 'rb') as f:
    meta = json.load(f)

all_rules = meta['rule_names']
direct_rules = meta['direct_rule_names']
topo_features = ['degrees', 'clustering', 'eigenvector_centrality', 'degree_centrality', 'avg_edge_weight']

In [34]:
meta.keys()

dict_keys(['nodes', 'edges', 'edge_homophily', 'kg_genes', 'rule_names', 'direct_rule_names', 'patients', 'labels', 'n_kg_genes', 'n_rules', 'n_direct_rules'])

In [41]:
rule_feature = torch.cat(
    [data_direct.topo_features, data_direct.kg_scores],
    dim=1)
rule_name = topo_features+direct_rules

df_rules, df_samples = explain(model=model,
                               data=data_direct,
                               rule_feature=rule_feature,
                               rule_names=rule_name,
                               patients=meta['patients'],
                               metric_name='auroc')

Attributions shape: torch.Size([691, 190])
Attributions shape: torch.Size([691, 190])


In [42]:
df_rules

,rule,alpha,theta,mean_activation,std_activation,activation_rate,grad_rule_importance_prediction,importance_class_0,importance_class_1,Permutation_importance(true_label),Permutation_importance(pred_label),Integrated_Gradient_importance(true_label),Integrated_Gradient_importance(pred_label)
rule_name,,,,,,,,,,,,,
degrees,0,0.999377,-0.000508,0.486994,0.161264,0.280753,0.005132,0.006785,0.004662,-0.000318,0.0,0.001368,0.001250
clustering,1,1.000928,0.000845,0.483525,0.193104,0.356006,0.005453,0.003274,0.005864,-0.000064,0.0,0.001723,0.001885
eigenvector_centrality,2,1.000845,0.000862,0.478795,0.136938,0.178003,0.002352,0.000798,0.002447,-0.000055,0.0,0.000404,0.000480
degree_centrality,3,0.999815,0.000808,0.486691,0.161336,0.280753,0.007003,0.006470,0.006989,-0.000073,0.0,0.001697,0.001723
avg_edge_weight,4,1.000178,-0.000609,0.527331,0.098439,0.891462,0.003779,0.001414,0.004039,-0.000118,0.0,0.000338,0.000391
...,...,...,...,...,...,...,...,...,...,...,...,...,...
direct_PIN1_CTL,185,0.999999,0.000823,0.500569,0.226806,0.493488,0.002070,0.005536,0.001751,0.000045,0.0,0.001327,0.000803
chain_PTGS2_PTGS2_AD,186,0.999606,0.000821,0.502246,0.213006,0.499276,0.006910,0.003597,0.007955,-0.000055,0.0,0.002208,0.002438
chain_CST3_APP_CTL,187,0.999844,0.000869,0.500323,0.225665,0.493488,0.002489,0.013924,0.000815,0.000600,0.0,0.002787,0.001225


In [43]:
df_samples

,degrees,clustering,eigenvector_centrality,degree_centrality,avg_edge_weight,chain_SORL1_APP_CTL,chain_MME_APP_CTL,chain_CTSD_APP_AD,direct_BIN1_AD,direct_PICALM_AD,...,direct_PPP1CA_AD,direct_RARRES3_AD,direct_PIN1_CTL,chain_PTGS2_PTGS2_AD,chain_CST3_APP_CTL,chain_ABCA1_APP_CTL,chain_GRN_GRN_AD,gate,Label,Prediction
sample_id,,,,,,,,,,,,,,,,,,,,,
DCR00001_1,0.000020,0.000362,0.000218,0.000060,0.000349,0.000439,7.885745e-07,0.000670,0.001196,-0.000758,...,0.002268,-0.002874,0.000490,0.004244,-0.000010,-0.000270,0.001248,0.514837,1,1
DCR00004_1,0.000019,0.000451,-0.000112,0.000054,0.000284,0.001233,1.306042e-04,0.000772,0.001216,-0.000814,...,0.001509,-0.003679,0.000014,0.002065,0.000065,-0.000699,0.001216,0.520395,1,1
DCR00009_1,0.000024,-0.001017,-0.000050,0.000053,-0.000009,0.000540,6.078311e-05,0.001618,0.000737,0.000128,...,0.001797,-0.002389,-0.000213,0.000803,-0.000080,-0.000299,0.001740,0.518918,0,1
DCR00013_1,0.000018,0.000381,-0.000029,0.000058,0.000122,0.000289,1.155570e-05,0.000153,0.001092,-0.000119,...,0.001066,-0.002781,0.000276,0.004010,0.000020,-0.000418,0.001132,0.517018,0,1
DCR00014_1,0.000020,-0.003363,-0.001006,0.000055,-0.000245,0.000535,3.113514e-05,0.000959,0.000749,-0.000764,...,0.001518,-0.002853,-0.000172,0.003064,0.000228,-0.000296,0.002407,0.521031,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TLSMCI607_1,0.002621,-0.007197,0.000396,0.003200,-0.000264,-0.000237,-1.838170e-03,-0.001420,-0.005749,0.000203,...,-0.000177,0.004335,0.001467,0.000163,-0.000430,-0.000008,-0.000482,0.490696,1,1
TLSMCI608_1,-0.001222,-0.005657,0.000752,-0.001593,-0.000217,-0.001729,4.352468e-03,-0.003571,-0.002975,-0.000282,...,0.000426,0.001241,0.000156,-0.001145,-0.000553,0.000157,-0.002019,0.487356,1,1
TLSMCI609_1,0.000361,-0.001132,0.000174,0.000797,0.000201,0.001120,1.929452e-04,0.002303,0.000629,-0.000487,...,0.001732,-0.001619,-0.000414,-0.000834,0.000146,0.000288,0.002706,0.525628,1,1


In [44]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score
from captum.attr import IntegratedGradients
from typing import Dict, Optional, Tuple, Any, List

def get_rule_params(model: nn.Module, 
                    data: Any, 
                    rule_feature: torch.Tensor) -> Tuple[pd.DataFrame, Dict[str, torch.Tensor]]:
    """
    Extracts rule parameters, activation statistics, and gradient-based importance 
    metrics for the fuzzy rules in the model.

    Args:
        model (nn.Module): The trained Fuzzy GAT/GIN model.
        data (Any): PyG data object containing graph structure and features.
        rule_feature (torch.Tensor): The input fuzzy rule features.

    Returns:
        Tuple[pd.DataFrame, Dict]: A tuple containing a DataFrame with rule statistics 
        and the raw result dictionary from the model's forward pass.
    """
    model.eval()
    model.zero_grad(set_to_none=True)

    # Forward pass
    result = model(
        data.x,
        data.edge_index,
        getattr(data, 'edge_attr', None),
        rule_feature
    )

    logits = result["logits"]
    r = result["rules"]
    
    # Ensure rules tensor can track gradients for autograd
    assert r.requires_grad, "Rule tensor must require gradients for importance extraction."

    # ---------------------------------------------------------
    # Rule activation statistics
    # ---------------------------------------------------------
    r_detached = r.detach()
    mean_activation = r_detached.mean(dim=0)
    std_activation = r_detached.std(dim=0)
    activation_rate = (r_detached > 0.5).float().mean(dim=0)

    # ---------------------------------------------------------
    # Gradient × activation: class 0
    # ---------------------------------------------------------
    score_0 = logits[:, 0].sum()
    grad_0 = torch.autograd.grad(score_0, r, retain_graph=True)[0]
    importance_class_0 = (r_detached * grad_0.detach()).abs().mean(dim=0)

    # ---------------------------------------------------------
    # Gradient × activation: class 1
    # ---------------------------------------------------------
    score_1 = logits[:, 1].sum()
    grad_1 = torch.autograd.grad(score_1, r, retain_graph=True)[0]
    importance_class_1 = (r_detached * grad_1.detach()).abs().mean(dim=0)

    # ---------------------------------------------------------
    # Gradient × activation: predicted class
    # ---------------------------------------------------------
    predicted_class = logits.detach().argmax(dim=1)
    predicted_score = logits[
        torch.arange(logits.size(0), device=logits.device), 
        predicted_class
    ].sum()
    
    grad_pred = torch.autograd.grad(predicted_score, r)[0]
    importance_prediction = (r_detached * grad_pred.detach()).abs().mean(dim=0)

    # ---------------------------------------------------------
    # Alpha / theta extraction
    # ---------------------------------------------------------
    alpha = model.fuzzy_layer.alpha.detach().cpu().numpy()
    theta = model.fuzzy_layer.theta.detach().cpu().numpy()

    # ---------------------------------------------------------
    # DataFrame aggregation
    # ---------------------------------------------------------
    rule_df = pd.DataFrame({
        "rule": range(len(alpha)),
        "alpha": alpha,
        "theta": theta,
        "mean_activation": mean_activation.cpu().numpy(),
        "std_activation": std_activation.cpu().numpy(),
        "activation_rate": activation_rate.cpu().numpy(),
        "grad_rule_importance_prediction": importance_prediction.cpu().numpy(),
        "importance_class_0": importance_class_0.cpu().numpy(),
        "importance_class_1": importance_class_1.cpu().numpy(),
    })

    return rule_df, result


def test(true: torch.Tensor, logits: torch.Tensor) -> Dict[str, float]:
    """
    Computes classification metrics given ground truth labels and model logits.

    Args:
        true (torch.Tensor): Ground truth labels.
        logits (torch.Tensor): Raw, unnormalized predictions from the model.

    Returns:
        Dict[str, float]: Dictionary containing accuracy, F1 (macro/weighted), 
        AUROC, and AUPRC scores.
    """
    # Detach and move to CPU for sklearn compatibility
    true_np = true.cpu().numpy()
    pred_np = logits.argmax(dim=1).cpu().numpy()
    
    # Calculate probabilities from raw logits safely
    prob_np = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()

    metrics = {}
    metrics["accuracy"] = accuracy_score(true_np, pred_np)
    metrics["f1"] = f1_score(true_np, pred_np, zero_division=0)
    metrics["f1_macro"] = f1_score(true_np, pred_np, average="macro", zero_division=0)
    metrics["f1_weighted"] = f1_score(true_np, pred_np, average="weighted", zero_division=0)
    
    # Only compute AUROC/AUPRC if there are multiple classes present in the true labels
    try:
        metrics["auroc"] = roc_auc_score(true_np, prob_np)
        metrics["auprc"] = average_precision_score(true_np, prob_np)
    except ValueError:
        metrics["auroc"] = float('nan')
        metrics["auprc"] = float('nan')
        
    return metrics


@torch.no_grad()
def permutation_importance(model: nn.Module, 
                           data: Any, 
                           rule_features: torch.Tensor, 
                           labels: torch.Tensor, 
                           metric_name: str) -> torch.Tensor:
    """
    Calculates feature importance by measuring the performance drop when 
    a specific feature is randomly permuted.

    Args:
        model (nn.Module): The trained GNN model.
        data (Any): PyG data object.
        rule_features (torch.Tensor): The input fuzzy rule features.
        labels (torch.Tensor): Target labels to evaluate against (true or predicted).
        metric_name (str): The specific metric from test() to use for evaluation.

    Returns:
        torch.Tensor: A 1D tensor representing the importance score of each rule feature.
    """
    model.eval()

    baseline_out = model(
        x=data.x, 
        edge_index=data.edge_index, 
        edge_attr=getattr(data, 'edge_attr', None), 
        rule_features=rule_features
    )
    
    baseline_score = test(labels, baseline_out['logits'])[metric_name]
    importance = []

    for j in range(rule_features.size(1)):
        X_perm = rule_features.clone()

        # Shuffle feature j randomly across all nodes
        perm = torch.randperm(X_perm.size(0), device=X_perm.device)
        X_perm[:, j] = X_perm[perm, j]

        out_perm = model(
            x=data.x, 
            edge_index=data.edge_index, 
            edge_attr=getattr(data, 'edge_attr', None), 
            rule_features=X_perm
        )
        
        perm_score = test(labels, out_perm['logits'])[metric_name]

        # Larger drop = more important feature
        importance.append(baseline_score - perm_score)

    return torch.tensor(importance, device='cpu')


class RuleFeatureWrapper(nn.Module):
    """
    A minimal PyTorch wrapper module that exposes only the `rule_features` 
    argument to Captum's Integrated Gradients, keeping graph structure static.
    """
    def __init__(self, model: nn.Module, x: torch.Tensor, edge_index: torch.Tensor, edge_attr: Optional[torch.Tensor] = None):
        super().__init__()
        self.model = model
        self.x = x
        self.edge_index = edge_index
        self.edge_attr = edge_attr
        
    def forward(self, rule_features: torch.Tensor) -> torch.Tensor:
        out = self.model(
            x=self.x, 
            edge_index=self.edge_index, 
            edge_attr=self.edge_attr, 
            rule_features=rule_features
        )
        return out["logits"]


def captum_explain(model: nn.Module, 
                   data: Any, 
                   rule_feature: torch.Tensor, 
                   targets: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Explains model predictions using Captum's Integrated Gradients algorithm.

    Args:
        model (nn.Module): The trained GNN model.
        data (Any): PyG data object.
        rule_feature (torch.Tensor): Input fuzzy rules to attribute.
        targets (torch.Tensor): 1D tensor mapping nodes to their target explanation classes.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: The aggregated mean feature importances 
        and the raw node-level attributions.
    """
    model.eval()
    wrapper = RuleFeatureWrapper(
        model=model, 
        x=data.x, 
        edge_index=data.edge_index, 
        edge_attr=getattr(data, 'edge_attr', None)
    )

    ig = IntegratedGradients(wrapper)
    num_nodes = rule_feature.size(0)

    # Note: Using internal_batch_size=num_nodes strictly prevents GNN dimension mismatch
    attributions, _ = ig.attribute(
        inputs=rule_feature,
        target=targets,
        internal_batch_size=num_nodes,  
        n_steps=50, 
        return_convergence_delta=True
    )

    feature_importance = attributions.abs().mean(dim=0)
    
    return feature_importance.cpu(), attributions.cpu()


def explain(model: nn.Module, 
            data: Any, 
            rule_feature: torch.Tensor, 
            rule_names: List[str], 
            patients: List[str], 
            metric_name: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Comprehensive pipeline to aggregate various rule importance metrics 
    (Activation Gradients, Permutation, and Integrated Gradients) into formatted DataFrames.

    Args:
        model (nn.Module): The trained model.
        data (Any): PyG data object.
        rule_feature (torch.Tensor): Tensor of node-level rule features.
        rule_names (List[str]): List of readable names for each rule feature.
        patients (List[str]): Identifiers for each node/sample in the graph.
        metric_name (str): Metric to target for permutation importance (e.g., 'auroc').

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: 
            1. df: Global rule parameters and importance metrics.
            2. df_samples: Node-level (patient-level) attributions and predictions.
    """
    # Get base rule statistics and model outputs
    df, results = get_rule_params(model=model, data=data, rule_feature=rule_feature)
    
    true_labels = data.y.clone().detach()
    pred_labels = results['logits'].argmax(dim=1).detach()

    # Calculate permutation importances
    perm_true_importance = permutation_importance(
        model, data, rule_feature, true_labels, metric_name
    )
    perm_pred_importance = permutation_importance(
        model, data, rule_feature, pred_labels, metric_name
    )
    
    # Calculate integrated gradient importances
    ig_true_importance, true_attribution = captum_explain(
        model, data, rule_feature, true_labels
    )
    ig_pred_importance, pred_attribution = captum_explain(
        model, data, rule_feature, pred_labels
    )

    # Assemble Global Dataframe
    df['Permutation_importance(true_label)'] = perm_true_importance.numpy()
    df['Permutation_importance(pred_label)'] = perm_pred_importance.numpy()
    df['Integrated_Gradient_importance(true_label)'] = ig_true_importance.numpy()
    df['Integrated_Gradient_importance(pred_label)'] = ig_pred_importance.numpy()
    
    df['rule_name'] = rule_names
    df.set_index('rule_name', inplace=True)

    # Assemble Sample-level Dataframe
    df_samples = pd.DataFrame(pred_attribution.numpy(), columns=rule_names)
    df_samples['gate'] = results['gate'].abs().mean(dim=1).detach().cpu().numpy()
    df_samples['sample_id'] = patients
    df_samples['Label'] = true_labels.cpu().numpy()
    df_samples['Prediction'] = pred_labels.cpu().numpy()
    df_samples.set_index('sample_id', inplace=True)

    return df, df_samples

In [16]:
result_path = "../nesy_gnn/results/explain/AD_KG/ANMerge/4/FuzzyNN/results.pkl"
with open(result_path, 'rb') as f:
    result = pickle.load(f)

result

{'FuzzyFuzzyNN_topo': {'fold_test_metrics': [{'accuracy': 0.6406926406926406,
    'balanced_accuracy': 0.5,
    'f1': 0.7810026385224275,
    'f1_macro': 0.39050131926121373,
    'f1_weighted': 0.5003826428628539,
    'auroc': 0.44443992184956044,
    'auprc': 0.6038308970175224},
   {'accuracy': 0.6391304347826087,
    'balanced_accuracy': 0.5,
    'f1': 0.7798408488063661,
    'f1_macro': 0.38992042440318303,
    'f1_weighted': 0.49842002075885133,
    'auroc': 0.49635275797065814,
    'auprc': 0.6411256115030314},
   {'accuracy': 0.6391304347826087,
    'balanced_accuracy': 0.5,
    'f1': 0.7798408488063661,
    'f1_macro': 0.38992042440318303,
    'f1_weighted': 0.49842002075885133,
    'auroc': 0.4647569871322023,
    'auprc': 0.611391353730034}],
  'fuzzy_rules': [tensor([[0.4948, 0.4551, 0.4373, 0.4948, 0.5381],
           [0.4948, 0.4551, 0.5504, 0.4948, 0.5262],
           [0.4948, 0.6219, 0.5155, 0.4948, 0.4998],
           ...,
           [0.4110, 0.6219, 0.4279, 0.4110, 0.5